# In this notebook, I will interpret cross validation and bayesian optimization results for a variety of combinations of models and response variables

These results correspond to the folder `pipeline/results/5-25-26_initial_optimized_models`. These models are not final, meaning there are models that are more optimized than these.

Additionally, some of these models have errors that include containing the response variable in the features, making the prediction irrelevant. This explains the unusually low rmse values lower in the document.

In [2]:
import pickle
import numpy as np
import pandas as pd
from pyhere import here

example_results_dict_visualization = {
    'CVscore': np.float(rmse_value),
    'best_params': {
        'impute': imputer_chosen,
        'params(condensed)': params,
        'scale': scaler
    },
    'logCVscore': np.float(rmse_value),
    'logbest_params': {
        'impute': imputer_chosen,
        'params(condensed)': params,
        'scale': scaler
    }
}

In [2]:
# print(list(rf_lrat_results['best_params'])[1:-1])

Due to the setup of the results dictionary from the pickle file, in theory I can extract everything I need using dictionary methods like indexing and stuff, loop through all of them with a for loop and put those in a pandas dataframe of the results. Then, I can just sort by response variable, display the cv error and fit the best params.

The hard part will be getting the original pipeline to put the params to, but that should work with some f-string replacement working backwards.

In [3]:
results_list = []

for response in ["LRATIO", "LM", "L_BOL", "MASS", "DIAM", "SURF_DENS", "TEMP", "T_BOL"]:
    for space in ["xgblogratio_space.pkl", "rf_space.pkl", "rflogratio_space.pkl", "catboost_space.pkl", "xgboost_space.pkl", "tree_space.pkl", "catlogratio_space.pkl"]:
        space_name = space.split("_")[0]

        with open(here("pipeline/results", f"{space_name}_{response}_results.pkl"), 'rb') as file:
            results = pickle.load(file)

            param_values = []
            for param in list(results['best_params'])[1:-1]:
                param_values.append(results['best_params'][param])

            log_param_values = []
            for param in list(results['logbest_params'])[1:-1]:
                log_param_values.append(results['logbest_params'][param])

            row1 = {
                'log': 0,
                'var': response,
                'space': space_name,
                'CV_RMSE': results['CVscore'],
                'impute_strat': results['best_params']['impute'],
                'scale_strat': results['best_params']['scale'],
                'param_names': list(results['best_params'])[1:-1], 
                'param_values': param_values
            }
            row2 = {
                'log': 1,
                'var': response,
                'space': space_name,
                'CV_RMSE': results['logCVscore'],
                'impute_strat': results['logbest_params']['impute'],
                'scale_strat': results['logbest_params']['scale'],
                'param_names': list(results['logbest_params'])[1:-1],
                'param_values': log_param_values
            }

            results_list.append(row1)
            results_list.append(row2)

results_df = pd.DataFrame(results_list)


Once i get my results back, I can plug them into this to make them a list. then, I can easily find the best model for each response variable and trace that back to the pipeline used to fit it and fit that on the whole training set if I want to, to get final values for metrics/such.

In [20]:
results_df.head()

,log,var,space,CV_RMSE,impute_strat,scale_strat,param_names,param_values
0,0,LRATIO,xgblogratio,76.987287,KNNImputer(),StandardScaler(),"[model__colsample_bylevel, model__colsample_by...","[0.6982392834527765, 0.7587558767877389, 7.031..."
1,1,LRATIO,xgblogratio,0.053508,SimpleImputer(),StandardScaler(),"[model__colsample_bylevel, model__colsample_by...","[1.0, 1.0, 7.112319853366851e-09, 0.0354047463..."
2,0,LRATIO,rf,52.387638,SimpleImputer(),passthrough,"[model__bootstrap, model__max_depth, model__ma...","[True, 29, None, 2, 7, 198]"
3,1,LRATIO,rf,0.029016,SimpleImputer(),RobustScaler(),"[model__bootstrap, model__max_depth, model__ma...","[True, 10, None, 1, 2, 1000]"
4,0,LRATIO,rflogratio,53.391987,SimpleImputer(),passthrough,"[model__bootstrap, model__max_depth, model__ma...","[True, 13, None, 2, 3, 644]"


In [4]:
log_scale_df = results_df[results_df['log'] == 1]
norm_scale_df = results_df[results_df['log'] == 0]

In [24]:
best_log_scores = log_scale_df.groupby('var')['CV_RMSE'].min()
best_log_scores.head(10)

var
DIAM         0.013142
LM           0.017089
LRATIO       0.028276
L_BOL        0.037166
MASS         0.038134
SURF_DENS    0.004882
TEMP         0.000732
T_BOL        0.005062
Name: CV_RMSE, dtype: float64

In [ ]:
best_norm_scores = norm_scale_df.groupby('var')['CV_RMSE'].min()
best_norm_scores.head(10) 

var
DIAM            0.007809
LM              1.743563
LRATIO         52.387638
L_BOL        8551.337375
MASS          222.317630
SURF_DENS       0.039066
TEMP            0.018650
T_BOL           0.432864
Name: CV_RMSE, dtype: float64

In [15]:
temp_models_ordered = norm_scale_df[norm_scale_df['var']=='TEMP'].sort_values(by='CV_RMSE')
print(temp_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
log_temp_models_ordered = log_scale_df[log_scale_df['var']=='TEMP'].sort_values(by='CV_RMSE')
print(log_temp_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log   var        space   CV_RMSE                      impute_strat  \
96    0  TEMP  catlogratio  0.018650  SimpleImputer(strategy='median')   
90    0  TEMP     catboost  0.018870                       passthrough   
88    0  TEMP   rflogratio  0.046259                   SimpleImputer()   

         scale_strat  
96  StandardScaler()  
90    RobustScaler()  
88    RobustScaler()  


    log   var        space   CV_RMSE     impute_strat     scale_strat
91    1  TEMP     catboost  0.000732  SimpleImputer()     passthrough
97    1  TEMP  catlogratio  0.000804     KNNImputer()     passthrough
89    1  TEMP   rflogratio  0.002964  SimpleImputer()  RobustScaler()


In [13]:
log_surf_dens_models_ordered = log_scale_df[log_scale_df['var']=='SURF_DENS'].sort_values(by='CV_RMSE')
print(log_surf_dens_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

dens_models_ordered = norm_scale_df[norm_scale_df['var']=='SURF_DENS'].sort_values(by='CV_RMSE')
print(dens_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log        var        space   CV_RMSE     impute_strat     scale_strat
83    1  SURF_DENS  catlogratio  0.004882  SimpleImputer()  RobustScaler()
77    1  SURF_DENS     catboost  0.006394  SimpleImputer()     passthrough
75    1  SURF_DENS   rflogratio  0.007429  SimpleImputer()  RobustScaler()
    log        var        space   CV_RMSE     impute_strat       scale_strat
82    0  SURF_DENS  catlogratio  0.039066  SimpleImputer()  StandardScaler()
76    0  SURF_DENS     catboost  0.042312     KNNImputer()  StandardScaler()
80    0  SURF_DENS         tree  0.044328     KNNImputer()  StandardScaler()


In [12]:
tbol_models_ordered = norm_scale_df[norm_scale_df['var']=='T_BOL'].sort_values(by='CV_RMSE')
print(tbol_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

print("\n")
log_tbol_models_ordered = log_scale_df[log_scale_df['var']=='T_BOL'].sort_values(by='CV_RMSE')
print(log_tbol_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

     log    var       space   CV_RMSE                      impute_strat  \
102    0  T_BOL  rflogratio  0.432864  SimpleImputer(strategy='median')   
100    0  T_BOL          rf  0.435320  SimpleImputer(strategy='median')   
108    0  T_BOL        tree  0.461044  SimpleImputer(strategy='median')   

          scale_strat  
102       passthrough  
100       passthrough  
108  StandardScaler()  


     log    var       space   CV_RMSE                      impute_strat  \
103    1  T_BOL  rflogratio  0.005062  SimpleImputer(strategy='median')   
101    1  T_BOL          rf  0.005101  SimpleImputer(strategy='median')   
109    1  T_BOL        tree  0.006832                   SimpleImputer()   

          scale_strat  
103  StandardScaler()  
101    RobustScaler()  
109    RobustScaler()  


In [10]:
log_lm_models_ordered = log_scale_df[log_scale_df['var']=='LM'].sort_values(by='CV_RMSE')
print(log_lm_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
lm_models_ordered = norm_scale_df[norm_scale_df['var']=='LM'].sort_values(by='CV_RMSE')
print(lm_models_ordered.drop(columns=['param_names', 'param_values']).head(3))


    log var       space   CV_RMSE                      impute_strat  \
19    1  LM  rflogratio  0.017089  SimpleImputer(strategy='median')   
17    1  LM          rf  0.017870  SimpleImputer(strategy='median')   
25    1  LM        tree  0.021870  SimpleImputer(strategy='median')   

         scale_strat  
19  StandardScaler()  
17       passthrough  
25  StandardScaler()  


    log var       space   CV_RMSE     impute_strat       scale_strat
16    0  LM          rf  1.743563  SimpleImputer()    RobustScaler()
18    0  LM  rflogratio  1.854985  SimpleImputer()  StandardScaler()
24    0  LM        tree  1.886387     KNNImputer()    RobustScaler()


In [9]:
log_lrat_models_ordered = log_scale_df[log_scale_df['var']=='LRATIO'].sort_values(by='CV_RMSE')
print(log_lrat_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
lrat_models_ordered = norm_scale_df[norm_scale_df['var']=='LRATIO'].sort_values(by='CV_RMSE')
print(lrat_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log     var       space   CV_RMSE     impute_strat     scale_strat
5     1  LRATIO  rflogratio  0.028276  SimpleImputer()  RobustScaler()
3     1  LRATIO          rf  0.029016  SimpleImputer()  RobustScaler()
11    1  LRATIO        tree  0.037063     KNNImputer()     passthrough


   log     var       space    CV_RMSE     impute_strat     scale_strat
2    0  LRATIO          rf  52.387638  SimpleImputer()     passthrough
4    0  LRATIO  rflogratio  53.391987  SimpleImputer()     passthrough
6    0  LRATIO    catboost  58.083966      passthrough  RobustScaler()


In [17]:
log_diam_models_ordered = log_scale_df[log_scale_df['var']=='DIAM'].sort_values(by='CV_RMSE')
print(log_diam_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
diam_models_ordered = norm_scale_df[norm_scale_df['var']=='DIAM'].sort_values(by='CV_RMSE')
print(diam_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log   var       space   CV_RMSE     impute_strat       scale_strat
59    1  DIAM          rf  0.013142  SimpleImputer()    RobustScaler()
61    1  DIAM  rflogratio  0.013766  SimpleImputer()  StandardScaler()
67    1  DIAM        tree  0.016366     KNNImputer()       passthrough


    log   var       space   CV_RMSE                      impute_strat  \
58    0  DIAM          rf  0.007809  SimpleImputer(strategy='median')   
60    0  DIAM  rflogratio  0.008040  SimpleImputer(strategy='median')   
66    0  DIAM        tree  0.008601  SimpleImputer(strategy='median')   

         scale_strat  
58  StandardScaler()  
60  StandardScaler()  
66  StandardScaler()  


In [8]:
log_mass_models_ordered = log_scale_df[log_scale_df['var']=='MASS'].sort_values(by='CV_RMSE')
print(log_mass_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
mass_models_ordered = norm_scale_df[norm_scale_df['var']=='MASS'].sort_values(by='CV_RMSE')
print(mass_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log   var       space   CV_RMSE  impute_strat     scale_strat
47    1  MASS  rflogratio  0.038134  KNNImputer()  RobustScaler()
45    1  MASS          rf  0.038476  KNNImputer()     passthrough
53    1  MASS        tree  0.046599  KNNImputer()     passthrough


    log   var       space     CV_RMSE                      impute_strat  \
52    0  MASS        tree  222.317630  SimpleImputer(strategy='median')   
46    0  MASS  rflogratio  227.273867                   SimpleImputer()   
44    0  MASS          rf  228.848124  SimpleImputer(strategy='median')   

         scale_strat  
52  StandardScaler()  
46  StandardScaler()  
44  StandardScaler()  


In [16]:
log_lbol_models_ordered = log_scale_df[log_scale_df['var']=='L_BOL'].sort_values(by='CV_RMSE')
print(log_lbol_models_ordered.drop(columns=['param_names', 'param_values']).head(3))
print("\n")
lbol_models_ordered = norm_scale_df[norm_scale_df['var']=='L_BOL'].sort_values(by='CV_RMSE')
print(lbol_models_ordered.drop(columns=['param_names', 'param_values']).head(3))

    log    var       space   CV_RMSE                      impute_strat  \
31    1  L_BOL          rf  0.037166                      KNNImputer()   
33    1  L_BOL  rflogratio  0.037649                      KNNImputer()   
39    1  L_BOL        tree  0.045939  SimpleImputer(strategy='median')   

         scale_strat  
31  StandardScaler()  
33    RobustScaler()  
39    RobustScaler()  


    log    var       space      CV_RMSE     impute_strat       scale_strat
32    0  L_BOL  rflogratio  8551.337375     KNNImputer()  StandardScaler()
30    0  L_BOL          rf  8836.240845  SimpleImputer()    RobustScaler()
38    0  L_BOL        tree  8836.550710  SimpleImputer()    RobustScaler()
